In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # comment this out if running interactively and you want inline plots
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NUMERIC_COLS = [
    "Age", "SibSp", "Parch", "Fare",
]
CATEGORICAL_COLS = [
    "Pclass", "Sex", "Embarked",
]

print("=" * 70)
print("TITANIC PASSENGER CLUSTERING: K-Means / Hierarchical / DBSCAN")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Load + preprocess
# ---------------------------------------------------------------------------
# >>> EDIT the filename below so it exactly matches your CSV file <
df = pd.read_csv("train.csv")

imputer = SimpleImputer(strategy="median")
num_df = pd.DataFrame(
    imputer.fit_transform(df[NUMERIC_COLS]), columns=NUMERIC_COLS, index=df.index
)

cat_source = df[CATEGORICAL_COLS].astype(str)
for col in CATEGORICAL_COLS:
    if df[col].isna().any():
        mode_val = df[col].mode()[0]
        cat_source[col] = cat_source[col].where(df[col].notna(), str(mode_val))
cat_df = pd.get_dummies(cat_source, drop_first=True)

feature_df = pd.concat([num_df, cat_df], axis=1)
X = StandardScaler().fit_transform(feature_df)

print(f"Loaded {len(df):,} passengers, {feature_df.shape[1]} clustering features "
      f"(Survived excluded)")

# ---------------------------------------------------------------------------
# 2. Choose k for K-Means via elbow + silhouette
# ---------------------------------------------------------------------------
k_range = range(2, 9)
sil_sample_size = min(5000, len(X))
sil_sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), sil_sample_size, replace=False)
X_sil_sample = X[sil_sample_idx]

inertias, sil_scores = [], []
for k in k_range:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = km_k.fit_predict(X)
    inertias.append(km_k.inertia_)
    sil_scores.append(silhouette_score(X_sil_sample, km_k.predict(X_sil_sample)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(list(k_range), inertias, marker="o")
ax1.set_title("Elbow method (inertia)")
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")

ax2.plot(list(k_range), sil_scores, marker="o", color="darkorange")
ax2.set_title("Silhouette score vs k")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette score")

plt.tight_layout()
plt.savefig("kmeans_elbow_silhouette.png", dpi=150)
print("Saved plot -> kmeans_elbow_silhouette.png")
plt.close(fig)

best_k = list(k_range)[int(np.argmax(sil_scores))]
print(f"Silhouette-suggested k = {best_k}  (scores: "
      + ", ".join(f"k={k}:{s:.3f}" for k, s in zip(k_range, sil_scores)) + ")")

# ---------------------------------------------------------------------------
# 3. K-Means on the FULL dataset
# ---------------------------------------------------------------------------
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
km_labels = kmeans.fit_predict(X)

sil = silhouette_score(X[sil_sample_idx], km_labels[sil_sample_idx])
print(f"\nK-Means (k={best_k}) silhouette (sample): {sil:.3f}")
print("Cluster sizes:", pd.Series(km_labels).value_counts().sort_index().to_dict())

# --- Profile clusters against Survived (label used only for reporting) ---
profiled = feature_df.copy()
profiled["KMeansCluster"] = km_labels
profiled["Survived"] = df["Survived"].values

summary = profiled.groupby("KMeansCluster").agg(
    n_passengers=("Survived", "size"),
    survival_rate=("Survived", "mean"),
    avg_Age=("Age", "mean"),
    avg_Fare=("Fare", "mean"),
    avg_SibSp=("SibSp", "mean"),
    avg_Parch=("Parch", "mean"),
).round(3)

print("\n--- Cluster profiles (KMeansCluster) ---")
print(summary.to_string())
summary.to_csv("cluster_profiles.csv")
print("Saved table -> cluster_profiles.csv")

# ---------------------------------------------------------------------------
# 4. Hierarchical + DBSCAN on a subsample (tractable for O(n^2) methods)
# ---------------------------------------------------------------------------
sub_size = min(5000, len(X))
sub_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), sub_size, replace=False)
X_sub = X[sub_idx]

hc = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hc_labels = hc.fit_predict(X_sub)
sil_hc = silhouette_score(X_sub, hc_labels)
print(f"\nHierarchical (k={best_k}, subsample n={sub_size}) silhouette: {sil_hc:.3f}")

db = DBSCAN(eps=1.0, min_samples=10)
db_labels = db.fit_predict(X_sub)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int(np.sum(db_labels == -1))
print(f"DBSCAN (eps=1.0, min_samples=10, subsample n={sub_size}) found {n_db_clusters} clusters and {n_noise} noise points")

TITANIC PASSENGER CLUSTERING: K-Means / Hierarchical / DBSCAN
Loaded 891 passengers, 9 clustering features (Survived excluded)
Saved plot -> kmeans_elbow_silhouette.png
Silhouette-suggested k = 8  (scores: k=2:0.236, k=3:0.286, k=4:0.340, k=5:0.361, k=6:0.344, k=7:0.358, k=8:0.361)

K-Means (k=8) silhouette (sample): 0.361
Cluster sizes: {0: 179, 1: 51, 2: 76, 3: 280, 4: 94, 5: 127, 6: 47, 7: 37}

--- Cluster profiles (KMeansCluster) ---
               n_passengers  survival_rate  avg_Age  avg_Fare  avg_SibSp  avg_Parch
KMeansCluster                                                                      
0                       179          0.464   29.519    20.799      0.397      0.358
1                        51          0.549   24.363    30.898      0.549      2.333
2                        76          0.395   27.888    13.067      0.434      0.105
3                       280          0.179   27.580    10.438      0.200      0.043
4                        94          0.457   25.228   